# Assignment 2: Dell – Electronics Product Design Agent
### Shishir Deshpande, MSDS 442

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

load_dotenv() # this will read my secrets and API keys from the .env file
os.environ["USER_AGENT"] = "MSDS442-Assignment2-Deshpande"

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
from IPython.display import display, Markdown
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
dell_url_list = ["https://www.dell.com/en-us/shop/dell-laptops/inspiron-14-2-in-1-laptop/spd/inspiron-14-7440-2-in-1-laptop",
"https://www.dell.com/en-us/shop/dell-laptops/latitude-5450-laptop/spd/latitude-14-5450-laptop",
"https://www.dell.com/en-us/shop/dell-laptops/latitude-7450-laptop/spd/latitude-14-7450-2-in-1-laptop",
"https://www.dell.com/en-us/shop/dell-laptops/xps-14-laptop/spd/xps-14-9440-laptop"]

In [ ]:
# Inspecting if webbaseloader can load content
for url in dell_url_list:
    loader = WebBaseLoader(url) # this is the loader that will fetch the HTML in the URL and wrap it to make it ready for consumption
    raw_docs = loader.load() # This will return a list of objects to feed into the LLM

    # Confirming inputs are processed
    print(f"Number of documents loaded: {len(raw_docs)}") # this will confirm how many documents got loaded
    print(f"Character count: {len(raw_docs[0].page_content)}") # this will confirm how much content got loaded in the first document
    print(raw_docs[0].page_content[:100]) # this will help me check what the content is about

In [ ]:
loader = WebBaseLoader(web_paths=dell_url_list) # basically loads it all at once
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, 
    chunk_overlap = 200
)

split_docs = text_splitter.split_documents(raw_docs)

print(f"Pages loaded: {len(raw_docs)}")
print(f"Total chunks created: {len(split_docs)}")
print(f"Sample chunk:\n{split_docs[0].page_content[:500]}")

In [ ]:
# Building a vector store to embed each chunk based on k-nearest score (this will help make relevant results rank higher)
vectorstore = Chroma.from_documents(
    documents = split_docs,
    embedding = OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 10} # will return top 10 most relevant chunks based on query
)

print("Vector store built successfully.")
print(f"Total chunks indexed: {vectorstore._collection.count()}")

### Building the salesperson agent

I will build the agent with two nodes: (1) Retrieving relevant chunks, (2) Generate a recommendation

In [ ]:
from langgraph.graph import START
from langchain_core.documents import Document

# Defining the schema for the agent flow
class SalespersonState(TypedDict):
    question: str
    retrieved_docs: list[Document]
    recommendation: str

# Node 1: Retrieve top 10 relevant chunks from vector store
def retrieve(state: SalespersonState) -> SalespersonState:
    docs = retriever.invoke(state["question"])
    return {"retrieved_docs": docs}

# Node 2: Generate recommendations using LLM but grounded in retrieved product information
salesperson_prompt = PromptTemplate.from_template("""
You are a knowledgeable and reliable Dell salesperson. Based on the customer's requirements and the product information below, you should recommend the most suitable Dell laptop models. Be very specific about why you are suggesting the model and whether it meets their needs. If there are multiple models that match the needs, you should recommend the best fit and then provide alternative suggestions. Please base your recommendation only on the product information provided. IMPORTANT: Only recommend models from this list: "Dell Inspiron 14 2-in-1 Laptop", "Dell Latitude 5450 Laptop", "Dell Latitude 7450 Laptop or 2-in-1", "XPS 14 Laptop". If you cannot find something that matches, you are STRICTLY NOT ALLOWED to recommend models outside this list. 

Customer request: {question}

Product information: {context}

Provide a clear, specific recommendation.""")

def generate_recommendation(state: SalespersonState) -> SalespersonState:
    context = "\n\n".join([doc.page_content for doc in state["retrieved_docs"]])
    response = model.invoke(
        salesperson_prompt.format(question=state["question"], context = context)
    )
    return {"recommendation": response.content}

# Creating the graph and wiring the nodes
graph = StateGraph(SalespersonState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate_recommendation", generate_recommendation)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate_recommendation")
graph.add_edge("generate_recommendation", END)

salesperson_agent = graph.compile()
print("Salesperson agent has compiled successfully!")

In [ ]:
prompt_result1 = salesperson_agent.invoke({"question": "I want a dell computer for travel that has Intel® Core™ 7 150U."})
display(Markdown(prompt_result1["recommendation"]))

In [ ]:
prompt_result2 = salesperson_agent.invoke({"question": "I want a dell computer that has Intel® Core™ Ultra 5 135U vPro® and has 512 GB SSD."})
display(Markdown(prompt_result2["recommendation"]))

In [ ]:
prompt_result3 = salesperson_agent.invoke({"question": "I want a dell computer that has Intel® Core™ Ultra 7 165U vPro® and 1 TB SSD?"})
display(Markdown(prompt_result3["recommendation"]))

In [ ]:
prompt_result4 = salesperson_agent.invoke({"question": "I want a light weight XPS computer with Intel® Core™ Ultra 7 165U vPro® and 1 TB SSD."})
display(Markdown(prompt_result4["recommendation"]))